<a href="https://colab.research.google.com/github/aryan30-tp/Deep-learning/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)

In [2]:
train_url = "https://www.dropbox.com/scl/fi/m2yj95tccxmzin3mnna47/atis_train_data.csv?rlkey=p61rpu2mwxjcb1ypfh89qzull&st=omw3dkpu&dl=1"
test_url = "https://www.dropbox.com/scl/fi/d1zwrv2jslo7j93p68l75/atis_test_data.csv?rlkey=0kl75tt54i2ccau9m1fxevlhn&st=w7n0orza&dl=1"

In [3]:
df_train = pd.read_csv(train_url, index_col=0)
df_test = pd.read_csv(test_url, index_col=0)

In [4]:
df_train.head()

,query,intent,slot filling
0,i want to fly from boston at 838 am and arriv...,flight,O O O O O B-fromloc.city_name O B-depart_time...
1,what flights are available from pittsburgh to...,flight,O O O O O B-fromloc.city_name O B-toloc.city_...
2,what is the arrival time in san francisco for...,flight_time,O O O B-flight_time I-flight_time O B-fromloc...
3,cheapest airfare from tacoma to orlando,airfare,B-cost_relative O O B-fromloc.city_name O B-t...
4,round trip fares from pittsburgh to philadelp...,airfare,B-round_trip I-round_trip O O B-fromloc.city_...


In [6]:
pd.set_option('display.max_colwidth', None)
df_small= pd.DataFrame(columns=['query', 'intent', 'slot filling'])
j=0
for i in df_train.intent.unique():
  df_small.loc[j]= df_train[df_train.intent==i].iloc[0]
  j=j+1

In [7]:
df_small

,query,intent,slot filling
0,i want to fly from boston at 838 am and arrive in denver at 1110 in the morning,flight,O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day
1,what is the arrival time in san francisco for the 755 am flight leaving washington,flight_time,O O O B-flight_time I-flight_time O B-fromloc.city_name I-fromloc.city_name O O B-depart_time.time I-depart_time.time O O B-fromloc.city_name
2,cheapest airfare from tacoma to orlando,airfare,B-cost_relative O O B-fromloc.city_name O B-toloc.city_name
3,what kind of aircraft is used on a flight from cleveland to dallas,aircraft,O O O O O O O O O O B-fromloc.city_name O B-toloc.city_name
4,what kind of ground transportation is available in denver,ground_service,O O O O O O O O B-city_name
5,what 's the airport at orlando,airport,O O O O O B-city_name
6,which airline serves denver pittsburgh and atlanta,airline,O O O B-fromloc.city_name B-fromloc.city_name O B-fromloc.city_name
7,how far is it from orlando airport to orlando,distance,O O O O O B-fromloc.airport_name I-fromloc.airport_name O B-toloc.city_name
8,what is fare code h,abbreviation,O O O O B-fare_basis_code
9,how much does the limousine service cost within pittsburgh,ground_fare,O O O O B-transport_type O O O B-city_name


In [8]:
query_data_train= df_train['query'].values
slot_data_train= df_train['slot filling'].values

query_data_test= df_test['query'].values
slot_data_test= df_test['slot filling'].values

In [9]:
max_query_length=30

In [10]:
#define text vectorisation layer
text_vectorisation_query= keras.layers.TextVectorization(
    output_sequence_length= max_query_length
)

#run training corpus to create vocab

text_vectorisation_query.adapt(query_data_train)
query_vocab_size= text_vectorisation_query.vocabulary_size()

In [11]:
#vectorize trainn and test queries

In [12]:
source_train= text_vectorisation_query(query_data_train)
source_test= text_vectorisation_query(query_data_test)

In [15]:
text_vectorisation_slots= keras.layers.TextVectorization(
    output_sequence_length= max_query_length,
    standardize=None
)
text_vectorisation_slots.adapt(slot_data_train)
slot_vocab_size=text_vectorisation_slots.vocabulary_size()


target_train= text_vectorisation_slots(slot_data_train)
target_test= text_vectorisation_slots(slot_data_test)

In [16]:
# adapted from https://keras.io/examples/nlp/text_classification_with_transformer/


from keras import ops
from keras import layers


class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads,
                                             key_dim=embed_dim,
                                             output_shape=embed_dim)
        self.ffn = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=maxlen, step=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [18]:
embed_dim=512
dense_dim=64
num_heads=5

units=128

In [19]:
embedding=TokenAndPositionEmbedding(
    max_query_length,
    query_vocab_size,
    embed_dim
)

In [20]:
te = TransformerEncoder(embed_dim,
                        dense_dim,
                        num_heads)

In [28]:
model_input= keras.layers.Input(shape=(max_query_length,))
x= embedding(model_input)
encoder_out= te(x)
x= keras.layers.Dense(128, activation='relu')(encoder_out)
x= keras.layers.Dropout(0.5)(x)
output= keras.layers.Dense(slot_vocab_size, activation="softmax")(x)
model= keras.Model(model_input,output)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 30, 512)        │       470,016 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, 30, 512)        │     5,319,232 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 30, 128)        │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 30, 125)        │        16,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,871,037 (22.40 MB)

 Trainable params: 5,871,037 (22.40 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',  metrics= ['sparse_categorical_accuracy'])

In [25]:
hist= model.fit(source_train, target_train, epochs=10, batch_size=64)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 232s 3s/step - loss: 0.1659 - sparse_categorical_accuracy: 0.9548
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 229s 3s/step - loss: 0.1117 - sparse_categorical_accuracy: 0.9648
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 246s 3s/step - loss: 0.0825 - sparse_categorical_accuracy: 0.9750
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 229s 3s/step - loss: 0.0651 - sparse_categorical_accuracy: 0.9805
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 243s 3s/step - loss: 0.0523 - sparse_categorical_accuracy: 0.9848
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 249s 3s/step - loss: 0.0417 - sparse_categorical_accuracy: 0.9879
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 267s 3s/step - loss: 0.0321 - sparse_categorical_accuracy: 0.9906
Epoch 8/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 242s 3s/step - loss: 0.0272 - sparse_categorical_accuracy: 0.9920
Epoch 9/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 256s 3s/step - loss: 0.0227 - sparse_categorical_accuracy: 0.9933
Epoch 10/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 297s 3s/step - loss: 0.01

In [26]:
model.evaluate(source_test,target_test)

28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 400ms/step - loss: 0.0677 - sparse_categorical_accuracy: 0.9878


[0.06772631406784058, 0.9877565503120422]

You can now use the following code cell to test the model interactively. Enter your query when prompted, and it will show you the predicted slot filling.

In [29]:
slot_vocab = text_vectorisation_slots.get_vocabulary()

def decode_slots(encoded_slots):
    return [slot_vocab[idx] for idx in encoded_slots]


while True:
    user_query = __builtins__.input("Enter your query (or type 'quit' to exit): ")
    if user_query.lower() == 'quit':
        break

    # Vectorize the user query
    vectorized_user_query = text_vectorisation_query([user_query])

    # Make a prediction with the model
    predictions = model.predict(vectorized_user_query, verbose=0)

    # Get the index of the highest probability for each token
    predicted_slot_indices = np.argmax(predictions, axis=-1)[0]

    # Decode the predicted slot indices back to labels
    predicted_slot_labels = decode_slots(predicted_slot_indices)

    print(f"Original Query: {user_query}")
    print(f"Predicted Slot Labels: {predicted_slot_labels}")
    print("\n" + "="*50 + "\n")

Enter your query (or type 'quit' to exit): how long are the fflights from new york to boston?
Original Query: how long are the fflights from new york to boston?
Predicted Slot Labels: [np.str_('B-depart_time.time_relative'), np.str_('B-flight_days'), np.str_('I-transport_type'), np.str_('B-arrive_date.day_number'), np.str_('B-depart_time.time_relative'), np.str_('I-mod'), np.str_('B-flight_days'), np.str_('B-flight_time'), np.str_('B-flight_days'), '', np.str_('I-depart_date.day_name'), np.str_('B-flight_days'), np.str_('B-return_date.month_name'), np.str_('I-depart_date.day_name'), np.str_('I-depart_date.day_name'), np.str_('B-flight_days'), np.str_('B-flight_days'), np.str_('B-flight_days'), np.str_('B-flight_days'), np.str_('I-depart_date.day_name'), np.str_('I-depart_date.day_name'), np.str_('I-depart_date.day_name'), np.str_('B-flight_days'), np.str_('B-flight_days'), np.str_('I-depart_date.day_name'), np.str_('B-return_date.month_name'), np.str_('B-flight_days'), np.str_('I-depar